# Publication-Quality Figure Generation for CSD-LLM Paper

This notebook demonstrates the evaluation artifact that generates **8 NeurIPS publication-quality figures** for the CSD-LLM (Critical Slowing Down in Large Language Models) paper.

**What it does:**
- Generates conceptual and schematic figures (Figs 1, 8) with NeurIPS styling (300 DPI, serif fonts, colorblind-safe palette, 3.25/6.75in column widths)
- Loads pre-computed evaluation metrics for all 8 figures (accuracy profiles, CSD dashboards, UMAP visualizations, classifier comparisons, model fits, temperature effects)
- Validates figure dimensions, file sizes, and NeurIPS compliance
- Visualizes evaluation results across all figures

**Figures overview:**
| Fig | Title | Type | Data |
|-----|-------|------|------|
| 1 | Conceptual Overview | conceptual | synthetic |
| 2 | Accuracy Profiles | empirical | experiment |
| 3 | CSD Indicator Dashboard | empirical | experiment |
| 4 | UMAP Flickering | visualization | experiment |
| 5 | Classifier Comparison | empirical | experiment |
| 6 | Model Fits | empirical | experiment |
| 7 | Temperature Effect | empirical | experiment |
| 8 | Prospective Protocol | conceptual | synthetic |

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'seaborn==0.13.2',
         'scipy==1.15.3', 'tabulate==0.9.0')


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import json
import sys
import os
import math
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np
from scipy import stats
from loguru import logger
from tabulate import tabulate
from IPython.display import display, HTML

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-276cb0-flickering-before-failing-ecological-cri/main/evaluation_iter5_publication_qua/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded evaluation data: {len(data['datasets'][0]['examples'])} figure results")
print(f"Scaling stage reached: {data['metadata']['scaling_stage_reached']}")
print(f"Total data points plotted: {data['metrics_agg']['total_data_points_plotted']}")

Loaded evaluation data: 8 figure results
Scaling stage reached: full
Total data points plotted: 856


In [5]:
# ── Configuration ──
# Tunable parameters — using full original values (fast enough for demo).

DPI = 300            # NeurIPS publication quality
SINGLE_COL = 3.25    # NeurIPS single-column width (inches)
DOUBLE_COL = 6.75    # NeurIPS double-column width (inches)
N_POINTS_CURVE = 500 # Points for conceptual curves
N_DIST_POINTS = 200  # Points for distribution curves

# Which self-contained figures to generate (1 and 8 need no external data)
FIGURES_TO_GENERATE = ["fig1", "fig8"]

PALETTE = sns.color_palette("colorblind")

print(f"Config: DPI={DPI}, curve_pts={N_POINTS_CURVE}, dist_pts={N_DIST_POINTS}")
print(f"Figures to generate: {FIGURES_TO_GENERATE}")

Config: DPI=300, curve_pts=500, dist_pts=200
Figures to generate: ['fig1', 'fig8']


## NeurIPS Style Setup

Configure matplotlib to match NeurIPS publication standards: serif fonts, appropriate sizes, and high-DPI output.

In [6]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Computer Modern Roman"],
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
})
print("NeurIPS style configured")

NeurIPS style configured


## Utility Functions

Helper functions from the original evaluation script for data access and numeric safety.

In [7]:
def get_dataset(data: dict, name: str) -> list:
    """Get a dataset by name from the datasets list."""
    for ds in data.get("datasets", []):
        if ds["dataset"] == name:
            return ds["examples"]
    raise KeyError(f"Dataset '{name}' not found in {[d['dataset'] for d in data.get('datasets', [])]}")


def safe_float(val, default: float = 0.0) -> float:
    """Convert a value to float, handling 'nan', None, etc."""
    if val is None:
        return default
    try:
        f = float(val)
        return default if math.isnan(f) else f
    except (ValueError, TypeError):
        return default


FIGURE_FILENAMES = {
    "fig1": "fig1_conceptual_overview",
    "fig2": "fig2_accuracy_profiles",
    "fig3": "fig3_csd_dashboard",
    "fig4": "fig4_umap_flickering",
    "fig5": "fig5_classifier_comparison",
    "fig6": "fig6_model_fits",
    "fig7": "fig7_temperature_effect",
    "fig8": "fig8_prospective_protocol",
}

FIGURE_CONFIGS = {
    "fig1": {"title": "Conceptual Overview", "type": "conceptual",
             "column": "double", "stage": "medium", "target_w": DOUBLE_COL},
    "fig2": {"title": "Accuracy Profiles", "type": "empirical",
             "column": "double", "stage": "mini", "target_w": DOUBLE_COL},
    "fig3": {"title": "CSD Indicator Dashboard", "type": "empirical",
             "column": "single", "stage": "mini", "target_w": SINGLE_COL},
    "fig4": {"title": "UMAP Flickering Visualization", "type": "visualization",
             "column": "double", "stage": "medium", "target_w": DOUBLE_COL},
    "fig5": {"title": "Classifier Comparison", "type": "empirical",
             "column": "single", "stage": "medium", "target_w": SINGLE_COL},
    "fig6": {"title": "Model Fits", "type": "empirical",
             "column": "single", "stage": "medium", "target_w": SINGLE_COL},
    "fig7": {"title": "Temperature Effect", "type": "empirical",
             "column": "single", "stage": "full", "target_w": SINGLE_COL},
    "fig8": {"title": "Prospective Protocol Schematic", "type": "conceptual",
             "column": "single", "stage": "full", "target_w": SINGLE_COL},
}

MODEL_SHORT = {
    "meta-llama/llama-3.1-8b-instruct": "Llama-3.1-8B",
    "google/gemini-2.0-flash-001": "Gemini Flash",
    "openai/gpt-4o-mini": "GPT-4o-mini",
    "google/gemini-2.0-flash-lite-001": "Gemini Flash Lite",
}

print(f"Defined {len(FIGURE_CONFIGS)} figure configs, {len(FIGURE_FILENAMES)} filenames")

Defined 8 figure configs, 8 filenames


## Figure 1: Conceptual Overview

Ecological CSD analog (left) showing water clarity vs nutrient loading with flickering zone, and LLM analog (right) showing response quality distributions transitioning from unimodal-correct through bimodal-flickering to unimodal-incorrect. This figure is fully self-contained (synthetic data).

In [8]:
def make_fig1():
    """Figure 1: Conceptual Overview -- Ecological + LLM analog."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(DOUBLE_COL, 3.0))

    # --- Panel (a): Ecological Analog ---
    x = np.linspace(0, 10, N_POINTS_CURVE)
    np.random.seed(42)

    # Piecewise clarity curve with flickering zone
    clarity = np.piecewise(
        x,
        [x < 3.5, (x >= 3.5) & (x < 6.5), x >= 6.5],
        [
            lambda d: 0.88 + 0.02 * np.sin(d * 3),
            lambda d: 0.55 + 0.20 * np.sin(d * 6) * np.exp(-((d - 5.0) ** 2) / 3.0),
            lambda d: 0.15 + 0.02 * np.sin(d * 3),
        ],
    )

    ax1.axvspan(0, 3.5, alpha=0.12, color="steelblue", zorder=0)
    ax1.axvspan(3.5, 6.5, alpha=0.12, color="gold", zorder=0)
    ax1.axvspan(6.5, 10, alpha=0.12, color="gray", zorder=0)

    ax1.plot(x, clarity, color="black", linewidth=0.9)
    ax1.set_xlabel("Nutrient Loading (Control Parameter)")
    ax1.set_ylabel("Water Clarity (State)")
    ax1.text(1.75, 0.05, "Clear", fontsize=7, ha="center", fontstyle="italic",
             color="steelblue")
    ax1.text(5.0, 0.05, "Flickering", fontsize=7, ha="center", fontstyle="italic",
             color="goldenrod")
    ax1.text(8.25, 0.05, "Turbid", fontsize=7, ha="center", fontstyle="italic",
             color="gray")
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 1.1)
    ax1.set_title("(a) Ecological CSD Analog", fontsize=8)
    ax1.tick_params(labelbottom=False, labelleft=False)

    # --- Panel (b): LLM Analog ---
    positions = [2, 5, 8]
    labels = ["Unimodal\nCorrect", "Flickering\n(Bimodal)", "Unimodal\nIncorrect"]
    colors_dist = [PALETTE[2], PALETTE[1], PALETTE[3]]

    for pos, label, color in zip(positions, labels, colors_dist):
        y_range = np.linspace(0, 1, N_DIST_POINTS)
        if pos == 2:
            dist = stats.norm.pdf(y_range, 0.85, 0.06)
        elif pos == 5:
            dist = (0.5 * stats.norm.pdf(y_range, 0.30, 0.08)
                    + 0.5 * stats.norm.pdf(y_range, 0.75, 0.08))
        else:
            dist = stats.norm.pdf(y_range, 0.15, 0.06)

        dist_scaled = dist / dist.max() * 0.9
        ax2.fill_betweenx(y_range, pos - dist_scaled, pos + dist_scaled,
                          alpha=0.45, color=color)
        ax2.plot(pos - dist_scaled, y_range, color=color, linewidth=0.7)
        ax2.plot(pos + dist_scaled, y_range, color=color, linewidth=0.7)
        ax2.text(pos, -0.13, label, fontsize=5.5, ha="center", va="top")

    ax2.axvspan(0, 3.5, alpha=0.06, color="steelblue", zorder=0)
    ax2.axvspan(3.5, 6.5, alpha=0.06, color="gold", zorder=0)
    ax2.axvspan(6.5, 10, alpha=0.06, color="gray", zorder=0)
    ax2.set_xlabel("Task Difficulty (Control Parameter)")
    ax2.set_ylabel("Response Quality Distribution")
    ax2.set_title("(b) LLM Analog", fontsize=8)
    ax2.set_xlim(0, 10)
    ax2.set_ylim(-0.18, 1.12)
    ax2.tick_params(labelbottom=False, labelleft=False)

    fig.tight_layout()
    return fig, 0

if "fig1" in FIGURES_TO_GENERATE:
    t0 = time.time()
    fig1, n1 = make_fig1()
    w, h = fig1.get_size_inches()
    print(f"Figure 1 generated: {w:.2f}x{h:.2f}in, {n1} data points ({time.time()-t0:.2f}s)")
    plt.show()
    plt.close(fig1)

Figure 1 generated: 6.75x3.00in, 0 data points (0.10s)


## Figure 8: Prospective Protocol Schematic

Horizontal flowchart showing the proposed CSD-based protocol: Sweep Difficulty -> Generate N Responses -> Compute CSD Indicators -> Threshold Test, with decision outputs (Flag near boundary vs Continue). This figure is fully self-contained (no external data).

In [9]:
def make_fig8():
    """Figure 8: Prospective Protocol Schematic -- horizontal flowchart."""
    fig, ax = plt.subplots(figsize=(SINGLE_COL, 2.5))
    ax.set_xlim(-0.2, 10.5)
    ax.set_ylim(-0.3, 3.3)
    ax.axis("off")

    # Boxes: (x, y, w, h, text, color)
    boxes = [
        (0.0, 1.0, 1.9, 1.0, "Sweep\nDifficulty", PALETTE[0]),
        (2.4, 1.0, 2.0, 1.0, "Generate N\nResponses", PALETTE[1]),
        (4.9, 1.0, 2.0, 1.0, "Compute CSD\nIndicators", PALETTE[2]),
        (7.5, 1.0, 1.6, 1.0, "Threshold\nTest", PALETTE[4]),
    ]

    for x, y, w, h, text, color in boxes:
        bbox = FancyBboxPatch(
            (x, y), w, h, boxstyle="round,pad=0.1",
            facecolor=(*color[:3], 0.25), edgecolor=color, linewidth=1.3,
        )
        ax.add_patch(bbox)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
                fontsize=5.5, fontweight="bold")

    # Arrows between boxes
    for i in range(len(boxes) - 1):
        x1 = boxes[i][0] + boxes[i][2]
        y1 = boxes[i][1] + boxes[i][3] / 2
        x2 = boxes[i + 1][0]
        y2 = boxes[i + 1][1] + boxes[i + 1][3] / 2
        ax.annotate(
            "", xy=(x2, y2), xytext=(x1, y1),
            arrowprops=dict(arrowstyle="-|>", color="gray", lw=1.5),
        )

    # Decision output: Flag (red)
    flag_x, flag_y = 9.5, 2.5
    ax.text(flag_x, flag_y, "Flag:\nNear Boundary", fontsize=5, ha="center",
            color="red", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="#ffcccc",
                      edgecolor="red", alpha=0.6))
    ax.annotate(
        "", xy=(flag_x - 0.3, flag_y - 0.3),
        xytext=(boxes[-1][0] + boxes[-1][2] / 2, boxes[-1][1] + boxes[-1][3]),
        arrowprops=dict(arrowstyle="-|>", color="red", lw=1.0),
    )

    # Decision output: Continue (green)
    cont_x, cont_y = 9.5, 0.3
    ax.text(cont_x, cont_y, "Continue", fontsize=5, ha="center",
            color="green", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="#ccffcc",
                      edgecolor="green", alpha=0.6))
    ax.annotate(
        "", xy=(cont_x - 0.3, cont_y + 0.2),
        xytext=(boxes[-1][0] + boxes[-1][2] / 2, boxes[-1][1]),
        arrowprops=dict(arrowstyle="-|>", color="green", lw=1.0),
    )

    # Annotations
    ax.text(5.9, 2.6, "variance  |  bimodality  |  dip", fontsize=4.5,
            ha="center", color="gray", fontstyle="italic",
            bbox=dict(facecolor="white", alpha=0.5, edgecolor="none"))

    fig.tight_layout()
    return fig, 0

if "fig8" in FIGURES_TO_GENERATE:
    t0 = time.time()
    fig8, n8 = make_fig8()
    w, h = fig8.get_size_inches()
    print(f"Figure 8 generated: {w:.2f}x{h:.2f}in, {n8} data points ({time.time()-t0:.2f}s)")
    plt.show()
    plt.close(fig8)

Figure 8 generated: 3.25x2.50in, 0 data points (0.04s)


## Evaluation Metrics Compilation

The original evaluation script validates each generated figure against NeurIPS publication standards. Below we compile and display the evaluation metrics from the loaded data, including figure dimensions, file sizes, data points plotted, and compliance checks.

In [10]:
# Extract figure results from loaded data
examples = get_dataset(data, "figure_generation_results")
metrics_agg = data["metrics_agg"]

# Build summary table
table_rows = []
for ex in examples:
    fig_id = ex["metadata_figure_id"]
    table_rows.append([
        fig_id,
        ex["input"].replace(f"Figure {fig_id[3:]}: ", ""),
        ex["metadata_figure_type"],
        ex["metadata_column_type"],
        f"{safe_float(ex['predict_width_inches']):.2f}x{safe_float(ex['predict_height_inches']):.2f}",
        f"{safe_float(ex['predict_filesize_png_kb']):.0f}",
        int(safe_float(ex["predict_n_data_points"])),
        "PASS" if ex["eval_size_check_pass"] == 1 else "FAIL",
        "PASS" if ex["eval_dimension_compliance"] == 1 else "FAIL",
    ])

headers = ["Fig", "Title", "Type", "Column", "Size (in)", "PNG KB", "Data Pts", "Size Check", "Dim Check"]
print(tabulate(table_rows, headers=headers, tablefmt="grid"))

print(f"\n--- Aggregate Metrics ---")
print(f"Figures generated: {metrics_agg['total_figures_generated']}/{metrics_agg['total_figures_attempted']}")
print(f"All pass size check: {'YES' if metrics_agg['all_figures_pass_size_check'] else 'NO'}")
print(f"NeurIPS dimension compliance: {'YES' if metrics_agg['neurips_dimension_compliance'] else 'NO'}")
print(f"Total data points plotted: {metrics_agg['total_data_points_plotted']}")
print(f"PNG size range: {metrics_agg['min_filesize_png_kb']:.0f} - {metrics_agg['max_filesize_png_kb']:.0f} KB "
      f"(mean: {metrics_agg['mean_filesize_png_kb']:.0f} KB)")

+-------+--------------------------------+---------------+----------+-------------+----------+------------+--------------+-------------+
| Fig   | Title                          | Type          | Column   | Size (in)   |   PNG KB |   Data Pts | Size Check   | Dim Check   |
+=======+================================+===============+==========+=============+==========+============+==============+=============+
| fig1  | Conceptual Overview            | conceptual    | double   | 6.75x3.00   |      164 |          0 | PASS         | PASS        |
+-------+--------------------------------+---------------+----------+-------------+----------+------------+--------------+-------------+
| fig2  | Accuracy Profiles              | empirical     | double   | 6.75x4.00   |      357 |        132 | PASS         | PASS        |
+-------+--------------------------------+---------------+----------+-------------+----------+------------+--------------+-------------+
| fig3  | CSD Indicator Dashboard        

## Results Visualization

Bar charts showing PNG file sizes and data points per figure, with NeurIPS size limit annotations and figure type color coding.

In [11]:
# --- Visualization of evaluation results ---
fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL + 2, 3.0))

fig_ids = [ex["metadata_figure_id"] for ex in examples]
fig_labels = [f"Fig {ex['metadata_figure_id'][3:]}" for ex in examples]
png_sizes = [safe_float(ex["predict_filesize_png_kb"]) for ex in examples]
data_points = [int(safe_float(ex["predict_n_data_points"])) for ex in examples]
fig_types = [ex["metadata_figure_type"] for ex in examples]

type_colors = {"conceptual": PALETTE[0], "empirical": PALETTE[1], "visualization": PALETTE[2]}
bar_colors = [type_colors.get(t, PALETTE[4]) for t in fig_types]

# Panel 1: PNG file sizes
ax1 = axes[0]
bars = ax1.bar(fig_labels, png_sizes, color=bar_colors, alpha=0.85)
ax1.axhline(y=2048, color="red", linestyle="--", linewidth=0.8, alpha=0.6, label="Max 2048 KB")
ax1.axhline(y=5, color="orange", linestyle="--", linewidth=0.8, alpha=0.6, label="Min 5 KB")
ax1.set_ylabel("PNG Size (KB)")
ax1.set_title("File Sizes", fontsize=9)
ax1.tick_params(axis="x", rotation=45)
ax1.legend(fontsize=5.5, loc="upper right")
for bar, val in zip(bars, png_sizes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f"{val:.0f}", ha="center", va="bottom", fontsize=5)

# Panel 2: Data points per figure
ax2 = axes[1]
bars2 = ax2.bar(fig_labels, data_points, color=bar_colors, alpha=0.85)
ax2.set_ylabel("Data Points")
ax2.set_title("Data Points Plotted", fontsize=9)
ax2.tick_params(axis="x", rotation=45)
for bar, val in zip(bars2, data_points):
    if val > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha="center", va="bottom", fontsize=5)

# Panel 3: Width compliance
ax3 = axes[2]
widths = [safe_float(ex["predict_width_inches"]) for ex in examples]
target_widths = [FIGURE_CONFIGS[ex["metadata_figure_id"]]["target_w"] for ex in examples]
x_pos = np.arange(len(fig_labels))
ax3.bar(x_pos - 0.15, widths, 0.3, label="Actual", color=PALETTE[0], alpha=0.85)
ax3.bar(x_pos + 0.15, target_widths, 0.3, label="Target", color=PALETTE[3], alpha=0.5)
ax3.set_xticks(x_pos)
ax3.set_xticklabels(fig_labels, rotation=45)
ax3.set_ylabel("Width (inches)")
ax3.set_title("Dimension Compliance", fontsize=9)
ax3.legend(fontsize=5.5)

# Legend for figure types
legend_elements = [mpatches.Patch(color=type_colors[t], alpha=0.85, label=t.capitalize())
                   for t in ["conceptual", "empirical", "visualization"]]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=7,
           bbox_to_anchor=(0.5, -0.05))

fig.suptitle("CSD-LLM Paper: 8-Figure Evaluation Summary", fontsize=10, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()
plt.close(fig)

print(f"\nAll {metrics_agg['total_figures_generated']} figures generated successfully with "
      f"{metrics_agg['total_data_points_plotted']} total data points.")


All 8 figures generated successfully with 856 total data points.
